# Многоагентная мафия — полный запуск в Kaggle

6 YandexGPT, 3 Gemini 3.5 Flash-Lite и 1 локальная GGUF-модель. Готовый llama.cpp без компиляции; T4 GPU, Internet и Secrets api_key, folder_id, google_api_key. Для выбранной публичной конверсии Sakura721 токен HF необязателен. При использовании авторизации разрешите hg_api_key.


План пьесы, рассказчик и архивная проверка включены. Выполняйте ячейки по порядку. Последняя основная ячейка начинает новую партию и делает реальные API-запросы; полученные файлы сохраняются в отдельный каталог. В конце есть ячейка продолжения после сбоя обычной дневной речи.


In [ ]:
# 1. Готовый официальный llama.cpp с CUDA 12.8. Без git, cmake и nvcc.
import hashlib, json, os, platform, re, shutil, socket, subprocess, sys, tarfile, time
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "requests"], check=True)
import requests
from huggingface_hub import HfApi, hf_hub_download

ROOT = Path("/kaggle/working/gguf_smoke")
ROOT.mkdir(parents=True, exist_ok=True)
CTX = 16384
REPO = "Sakura721/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"
FILE = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-Q4_K_M.gguf"
RELEASE = "b12110"
LLAMA_SHA = "cd959bae99ff4bed22bef2604f45241c09bbb8c8"
assert sys.platform == "linux" and platform.machine() == "x86_64", "Эта сборка предназначена для Kaggle Linux x64"
assert shutil.which("nvidia-smi"), "Включите GPU в настройках Kaggle"

def gpu_memory():
    return subprocess.check_output(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], text=True)

print("До загрузки модели:\n" + gpu_memory())
# Оба архива — из одного официального релиза. Второй содержит CUDA runtime и cuBLAS.
archives = {
    f"llama-{RELEASE}-bin-ubuntu-cuda-12.8-x64.tar.gz": "4af1e1da23661db4a5b84b29aae16ebc053df689c5b8b1427c4670f2c6406d8c",
    f"cudart-llama-{RELEASE}-bin-ubuntu-cuda-12.8-x64.tar.gz": "6f1d562086d84ae85df880d3711079ec225602b51d530608e5001600625067dd",
}
install = ROOT / "prebuilt" / RELEASE
install.mkdir(parents=True, exist_ok=True)
for name, expected_sha in archives.items():
    archive = install / name
    if not archive.exists():
        print("Скачивание:", name, flush=True)
        temporary = archive.with_suffix(".download")
        with requests.get(f"https://github.com/ggml-org/llama.cpp/releases/download/{RELEASE}/{name}",
                          stream=True, timeout=(20, 180)) as response:
            response.raise_for_status()
            with temporary.open("wb") as output:
                for block in response.iter_content(1024 * 1024):
                    output.write(block)
        temporary.replace(archive)
    with archive.open("rb") as file:
        actual_sha = hashlib.file_digest(file, "sha256").hexdigest()
    if actual_sha != expected_sha:
        archive.unlink()
        raise RuntimeError(f"SHA-256 не совпал для {name}; повторите ячейку для загрузки заново")
    with tarfile.open(archive) as bundle:
        bundle.extractall(install, filter="data")

BIN_DIR = install / f"llama-{RELEASE}"
CUDA_DIR = install / f"cudart-llama-{RELEASE}-bin-ubuntu-cuda-12.8-x64"
BINARY = BIN_DIR / "llama"
assert BINARY.is_file() and CUDA_DIR.is_dir(), "Неожиданная структура официального архива"
# Только окружение текущего ядра: готовый движок увидит приложенные CUDA-библиотеки.
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join([str(BIN_DIR), str(CUDA_DIR), os.environ.get("LD_LIBRARY_PATH", "")])
probe = subprocess.run([str(BINARY), "serve", "--list-devices"], capture_output=True, text=True,
                       env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print(probe.stdout + probe.stderr)
if probe.returncode != 0 or not re.search(r"CUDA\d+:", probe.stdout + probe.stderr):
    raise RuntimeError("Готовый движок не увидел CUDA GPU. Диагностика напечатана выше; веса ещё не загружались.")
print("Готово:", RELEASE, "— CUDA GPU обнаружена, компиляции не было")

In [ ]:
# 2. Скачать выбранный GGUF с явной авторизацией и проверкой доступа.
from pathlib import Path
import os
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

# Явно выбранная новая конверсия того же upstream; это не файл прежней партии.
REPO = "Sakura721/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"
FILE = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-Q4_K_M.gguf"
REVISION = "35285bf9d161fb3a9f28aee914670b0b32e72f3d"
HF_SECRET_NAME = "hg_api_key"  # Имя секрета Hugging Face в нашей Kaggle-сессии.

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME).strip() or False
except Exception:
    # token=False исключает неявное использование старого сохранённого токена.
    hf_token = os.environ.get("HF_TOKEN", "").strip() or False

print("Источник GGUF:", REPO)
print("Файл:", FILE)
api = HfApi(token=hf_token)
if hf_token:
    try:
        api.whoami()
    except HfHubHTTPError as exc:
        status = getattr(exc.response, "status_code", None)
        if status == 401:
            raise RuntimeError("HF-токен отклонён. Обновите секрет hg_api_key: нужен действующий токен Hugging Face.") from None
        raise RuntimeError(f"Не удалось проверить HF-токен (HTTP {status}); веса не скачивались.") from None

try:
    api.model_info(REPO, revision=REVISION)
except HfHubHTTPError as exc:
    status = getattr(exc.response, "status_code", None)
    if status in (401, 403, 404):
        detail = ("Токен действителен, но доступа к выбранному репозиторию/ревизии нет."
                  if hf_token else "Анонимный доступ недоступен. Включите Kaggle Secret hg_api_key и повторите ячейку.")
        raise RuntimeError(detail + " Проверьте страницу модели под своим HF-аккаунтом; веса автоматически не заменяются.") from None
    raise RuntimeError(f"Не удалось проверить репозиторий (HTTP {status}); веса не скачивались.") from None

MODEL = Path(hf_hub_download(REPO, FILE, revision=REVISION,
                           local_dir=ROOT / "models", token=hf_token))
print("Файл:", MODEL.name)
print("Ревизия:", REVISION)
print("Размер:", round(MODEL.stat().st_size / 1e9, 2), "GB")


In [ ]:
# 3. Загрузить модель на первую GPU и дождаться готовности сервера.
# --fit off запрещает автоуменьшение запрошенного offload при нехватке памяти.
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]
URL = f"http://127.0.0.1:{port}"
LOG = ROOT / "server.log"
args = [str(BINARY), "serve", "-m", str(MODEL), "-c", str(CTX),
        "--n-gpu-layers", "all", "--fit", "off", "--split-mode", "none", "--device", "CUDA0",
        "--parallel", "1", "-b", "128", "-ub", "128", "--flash-attn", "off",
        "--chat-template-kwargs", '{"enable_thinking":false}',
        "--host", "127.0.0.1", "--port", str(port)]
with LOG.open("w") as log:
    server = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
                              env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
try:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError("Сервер завершился. Последние строки:\n" + LOG.read_text(errors="replace")[-6000:])
        try:
            if requests.get(URL + "/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise TimeoutError("Модель не загрузилась за 300 секунд; проверьте " + str(LOG))
except BaseException:
    if server.poll() is None:
        server.terminate()
        server.wait(timeout=30)
    raise
print("Модель загружена:", URL)
print(gpu_memory())  # Общая занятая память GPU, не только память весов.
for line in LOG.read_text(errors="replace").splitlines():
    if any(s in line.lower() for s in ("offloaded", "model buffer size", "kv buffer size")):
        print(line)

## Актуальный движок мафии и настройки

In [ ]:
# Распаковка собственного кода проекта; веса и ключи в пакет не входят.
import base64, io, json, os, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from IPython.display import Markdown, display, FileLink
ROOT = (Path("/kaggle/working/mafia_demo") if Path("/kaggle/working").exists()
        else Path.cwd() / "runs" / "mafia_notebook")
ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE))) as bundle:
    bundle.extractall(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Повторный запуск в том же kernel должен импортировать новый снимок правил.
import importlib
for module_name in list(sys.modules):
    if module_name == "mafia_lab" or module_name.startswith("mafia_lab."):
        sys.modules.pop(module_name)
importlib.invalidate_caches()
from mafia_lab.mafia.models import Config
from mafia_lab.mafia.rules import RULES
from mafia_lab.mafia.engine import Game
from mafia_lab.mafia.story import StoryDirector
from mafia_lab.mafia.agent import ModelPlayer
from mafia_lab.mafia.output import Writer, render
from mafia_lab.mafia.llm import LlamaCppModel
from mafia_lab.mafia.gemini import GeminiModel
from mafia_lab.mafia.providers import player_models
from mafia_lab.mafia.credentials import read_credential
from mafia_lab.llm import YandexGPTModel
print(RULES)

In [ ]:
MODE = "live"  # example: сохранённая запись без новых API; live: новая партия
USE_GGUF = True
USE_GEMINI = True
GEMINI_PLAYER_IDS = ["p2", "p6", "p9"]
GEMINI_MODEL = "gemini-3.5-flash-lite"  # Проверена через API, включая игровые JSON-схемы.
LOCAL_PLAYER_ID = "p5"  # Дина. Роль случайна, её задаёт seed, а не модель.
# URL остаётся от нашего теста GGUF. Указывается адрес API с окончанием /v1.
GGUF_BASE_URL = (globals().get("URL", "").rstrip("/") + "/v1") if globals().get("URL") else ""
USE_STORY = True  # План и ремарки: дополнительные запросы YandexGPT
ARCHIVE_REVIEW = True
config = Config(seed=17, max_days=30, archive_review=ARCHIVE_REVIEW)
if MODE not in {"example", "live"}:
    raise ValueError("MODE: example или live")
if LOCAL_PLAYER_ID not in {f"p{i}" for i in range(1, 11)}:
    raise ValueError("LOCAL_PLAYER_ID: p1..p10")

In [ ]:
from IPython.display import display, FileLink
import zipfile
_progress = display({"text/plain":"Подготовка смешанной партии..."}, raw=True, display_id=True)

class ProgressWriter(Writer):
    def save(self, config, state):
        super().save(config, state)
        calls = sum(e.kind == "llm_response" for e in self.journal.events)
        errors = sum(e.kind == "validation_error" for e in self.journal.events)
        line = (f"MAFIA {state.status} | day={state.day} phase={state.phase} "
                f"alive={sum(p.alive for p in state.players)} calls={calls} fixes={errors}")
        if state.events and state.events[-1].kind != "rules":
            last = state.events[-1]
            line += f"\n{last.id} {last.kind}: {last.text[:140]}"
        _progress.update({"text/plain":line}, raw=True)

import importlib.util
import subprocess
from urllib.request import urlopen
if importlib.util.find_spec("openai") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "openai>=1.68,<3"], check=True)

api_key = read_credential("YANDEX_API_KEY")
folder = read_credential("YANDEX_FOLDER_ID")
model = YandexGPTModel("yandexgpt/latest", folder, api_key,
                       timeout=60, max_output_tokens=1800, temperature=0.5, max_retries=2)
gemini = GeminiModel(GEMINI_MODEL, read_credential("GOOGLE_API_KEY"), max_retries=5) if USE_GEMINI else None
local_model = None
if USE_GGUF:
    if not GGUF_BASE_URL:
        raise RuntimeError("Укажите GGUF_BASE_URL: URL работающего llama.cpp с окончанием /v1")
    with urlopen(GGUF_BASE_URL.rstrip("/") + "/models", timeout=10) as response:
        model_id = json.load(response)["data"][0]["id"]
    local_model = LlamaCppModel(model_id, GGUF_BASE_URL,
                                options={"max_tokens": 512, "temperature": 0.5}, timeout=180)

output_dir = ROOT / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6])
writer = ProgressWriter(output_dir, ROOT / "chat-template.html")
models = player_models(model, gemini=gemini, gemini_ids=GEMINI_PLAYER_IDS if USE_GEMINI else [],
                       local=local_model, local_id=LOCAL_PLAYER_ID)
writer.player_models = {pid:getattr(backend, "model", backend.name) for pid,backend in models.items()}
agents = {pid: ModelPlayer(backend, writer.journal, config.attempts) for pid, backend in models.items()}
director = StoryDirector(model, writer.journal, config.attempts) if USE_STORY else None
game = Game(config, agents, writer, director=director)


provenance = {"models":{pid:getattr(model, "model", model.name) for pid,model in models.items()},
              "source":"Mafia_Kaggle_Live.ipynb", "synthetic":False,
              "gguf_repository":REPO, "gguf_filename":FILE,
              "gguf_revision":REVISION, "gguf_context":CTX}
(output_dir / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
print("Результаты:", output_dir)
try:
    game.run()
except Exception:
    print("Партия остановлена; ошибка и частичный результат сохранены.")
    raise
finally:
    archive_path = Path("/kaggle/working/mafia_public.zip")
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for filename in ("public.json", "public.md", "summary.json", "chat.html", "table.html", "provenance.json"):
            path = output_dir / filename
            if path.exists(): archive.write(path, filename)
    print("Публичная запись:")
    display(FileLink(str(archive_path)))
print((output_dir / "summary.json").read_text())
display(FileLink(str(output_dir / "chat.html")))
display(FileLink(str(output_dir / "table.html")))

## Продолжение после сбоя дневной речи

Если обычная речь прервалась из-за API, выполните следующую ячейку в том же ядре. Принятые речи и назначения моделей сохраняются. Другие фазы этим способом не продолжаются.

In [ ]:
if "game" in globals() and game.state.status == "failed" and game.state.phase == "discussion":
    # Выполнить как одну новую ячейку в том же Kaggle kernel после сбоя Gemini 503.
    # Не запускает новую партию и не меняет модели участников.
    """Продолжение после сбоя обычной дневной речи; другие фазы не переигрываем."""
    from pathlib import Path
    from uuid import uuid4
    from mafia_lab.mafia.voting import Voting


    def finish_day(game):
        s = game.state
        if not s.nominees or (s.day == 1 and len(s.nominees) == 1):
            game.emit('no_vote', 'Голосования нет: '+(
                'никого не выставили.' if not s.nominees else 'единственный кандидат в первый день.'))
            return
        game.eliminate(Voting(game).run(list(s.nominees)))
        game.finish_if_won()


    def resume_discussion(game):
        """Только живой объект game в том же kernel, после ошибки обычной речи."""
        s = game.state
        if s.status != 'failed' or s.phase != 'discussion' or not s.prepared:
            raise ValueError('Продолжение поддержано только после сбоя обычной дневной речи, в том же ядре')
        living = [p.id for p in s.players if p.alive]
        if s.first_speaker_id not in living or not 1 <= s.day <= game.config.max_days:
            raise ValueError('Некорректная точка продолжения')
        start = living.index(s.first_speaker_id)
        order = living[start:] + living[:start]
        done = [e.speaker_id for e in s.events if e.day == s.day and e.kind == 'speech']
        if done != order[:len(done)]:
            raise ValueError('Записанные речи не совпадают с началом текущего круга')
        if any(e.day == s.day and e.kind in ('ballot_candidate','vote','elimination') for e in s.events):
            raise ValueError('Голосование уже начиналось; повторять его этим способом нельзя')
        # Сохранить точный снимок до смены статуса. Секреты остаются в приватном файле.
        directory = Path(game.writer.directory)
        game.writer.save(game.config, s)
        backup = directory / ('private_before_resume_'+uuid4().hex[:10]+'.json')
        backup.touch(mode=0o600)
        backup.write_bytes((directory/'private_result.json').read_bytes())
        backup.chmod(0o600)
        game.writer.journal.emit('run_recovery', {'day':s.day, 'phase':s.phase,
            'accepted_speakers':done, 'pending_speakers':order[len(done):], 'backup':backup.name})
        s.status, s.error = 'running', None
        game.writer.save(game.config, s)
        try:
            for pid in order[len(done):]:
                game.speech(pid)
            finish_day(game)
            if s.status == 'running':
                game.night()
            for day in range(s.day+1, game.config.max_days+1):
                if s.status != 'running':
                    break
                s.day = day
                game.day()
                if s.status == 'running':
                    game.night()
            if s.status == 'running':
                s.phase, s.status = 'finished', 'limited'
                game.emit('limit', 'Партия прервана по техническому лимиту дней; игровой результат не определён.')
        except (Exception, KeyboardInterrupt) as exc:
            s.status, s.error = 'failed', str(exc) or type(exc).__name__
            raise
        finally:
            game.writer.save(game.config, s)
        return s


    # У существующих объектов остаются те же провайдеры и учётные данные.
    if "game" not in globals():
        raise RuntimeError("Нет объекта game. Нужен тот же kernel, в котором произошёл сбой.")
    for player in game.agents.values():
        backend = getattr(getattr(player, "agent", None), "model", None)
        if backend is not None and getattr(backend, "name", "").startswith("google/"):
            backend.max_retries = 5

    print("Продолжаем с дневного обсуждения; принятые речи не повторяются.")
    try:
        resume_discussion(game)
    finally:
        # Обновить публичный архив и при повторной ошибке, не включая тайные роли/ключи.
        import zipfile
        from IPython.display import display, FileLink
        archive_path = Path("/kaggle/working/mafia_public.zip")
        with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
            for filename in ("public.json", "public.md", "summary.json", "chat.html", "table.html", "provenance.json"):
                path = game.writer.directory / filename
                if path.exists():
                    archive.write(path, filename)
        print("Публичная запись:", archive_path)
        display(FileLink(str(archive_path)))
    print((game.writer.directory/"summary.json").read_text())

else:
    print("Продолжение не требуется либо фаза не поддерживается.")